# Chagatai Stanza training from the unified dataset

This notebook consumes only the schema 2 canonical dataset. Auxiliary languages are train-only; Chagatai dev and test remain isolated.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

if importlib.util.find_spec('stanza') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'stanza'])

from stanza.models import tokenizer

print('Python:', sys.version)


In [ ]:
def find_unified_build() -> Path:
    explicit = os.environ.get('UNIFIED_DATASET_DIR')
    if explicit:
        candidates = [Path(explicit)]
    else:
        candidates = [Path('data/UNIFIED/builds/chagatai_uzs_uyghur_balanced')]
        kaggle_root = Path('/kaggle/input')
        if kaggle_root.exists():
            candidates.extend(path.parent for path in kaggle_root.glob('**/manifest.json'))

    valid = []
    for candidate in candidates:
        required = [candidate / name for name in ('manifest.json', 'source_sentences.csv', 'train.csv', 'dev.csv', 'test.csv')]
        if all(path.exists() for path in required):
            manifest = json.loads((candidate / 'manifest.json').read_text(encoding='utf-8'))
            if manifest.get('schema_version') == '2.0':
                valid.append(candidate)
    if len(valid) != 1:
        raise RuntimeError(f'Expected exactly one schema 2.0 unified build; found: {valid}')
    return valid[0]

UNIFIED_DIR = find_unified_build()
WORK_DIR = Path('/kaggle/working/chagatai_sbd') if Path('/kaggle/working').exists() else Path('work/chagatai_sbd')
STANZA_DIR = WORK_DIR / 'stanza'
MODEL_DIR = WORK_DIR / 'models'
STANZA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads((UNIFIED_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['checks']['status'] == 'passed'
assert manifest['checks']['auxiliary_languages_are_train_only']
sources = pd.read_csv(UNIFIED_DIR / 'source_sentences.csv', dtype=str).fillna('')
assert set(sources.loc[sources['language'] != 'chg', 'split']) <= {'train'}
for split in ('dev', 'test'):
    frame = pd.read_csv(UNIFIED_DIR / f'{split}.csv', dtype=str).fillna('')
    assert set(frame['language']) == {'chg'}
    assert set(frame['method']) == {'sequential'}
print('Unified build:', UNIFIED_DIR)
print('Source counts:', manifest['source_counts'])


In [ ]:
def stanza_text_and_labels(tokens: list[str], word_labels: list[int]) -> tuple[str, str]:
    chars = []
    labels = []
    for token_index, (token, word_label) in enumerate(zip(tokens, word_labels)):
        if token_index:
            chars.append(' ')
            labels.append('0')
        for char_index, char in enumerate(token):
            chars.append(char)
            labels.append('2' if word_label and char_index == len(token) - 1 else '1' if char_index == len(token) - 1 else '0')
    return ''.join(chars), ''.join(labels)

def export_split(split: str) -> int:
    frame = pd.read_csv(UNIFIED_DIR / f'{split}.csv', dtype=str).fillna('')
    texts = []
    labels = []
    for row in frame.itertuples(index=False):
        tokens = json.loads(row.tokens)
        word_labels = json.loads(row.labels)
        text, label_text = stanza_text_and_labels(tokens, word_labels)
        assert text == row.text
        assert len(text) == len(label_text)
        texts.append(text)
        labels.append(label_text)
    (STANZA_DIR / f'{split}.txt').write_text('\n\n'.join(texts) + '\n\n', encoding='utf-8')
    (STANZA_DIR / f'{split}.toklabels').write_text('\n\n'.join(labels) + '\n\n', encoding='utf-8')
    return len(frame)

counts = {split: export_split(split) for split in ('train', 'dev', 'test')}
(STANZA_DIR / 'mwt.json').write_text('[]\n', encoding='utf-8')
print('Stanza rows:', counts)


In [ ]:
RUN_TRAINING = False
DEVICE = 'cuda'
MODEL_NAME = 'chg_unified_tokenizer.pt'

if RUN_TRAINING:
    args = [
        '--txt_file', str(STANZA_DIR / 'train.txt'),
        '--label_file', str(STANZA_DIR / 'train.toklabels'),
        '--dev_txt_file', str(STANZA_DIR / 'dev.txt'),
        '--dev_label_file', str(STANZA_DIR / 'dev.toklabels'),
        '--mwt_json_file', str(STANZA_DIR / 'mwt.json'),
        '--lang', 'chg', '--shorthand', 'chg_unified',
        '--save_dir', str(MODEL_DIR), '--save_name', MODEL_NAME,
        '--mode', 'train', '--device', DEVICE,
        '--steps', '30000', '--eval_steps', '200', '--report_steps', '50',
        '--max_steps_before_stop', '3000', '--batch_size', '32', '--seed', '42',
    ]
    tokenizer.main(args)
else:
    print('Set RUN_TRAINING=True after reviewing the manifest and counts above.')
